In [1]:
%AddJar file:/app/scala_lib/build/libs/robustInfer-scala-0.1.0.jar

Starting download from file:/app/scala_lib/build/libs/robustInfer-scala-0.1.0.jar
Finished download of robustInfer-scala-0.1.0.jar
Using cached version of robustInfer-scala-0.1.0.jar


In [2]:
import robustinfer._

In [3]:
import org.apache.spark.sql.{Dataset, Encoder, Encoders,SparkSession}
import org.apache.spark.sql.functions._
import scala.util.Random
import org.apache.spark.rdd.RDD

## generate a small data

In [4]:
// manuall generate a small dataset
import breeze.linalg.{DenseVector, norm}

val smallData = Seq(
  Obs(i="1", x=Array(1.0, 2.0), y=1.0, timeIndex=Some(1), z=Some(1.0)),
  Obs(i="2", x=Array(2.0, 3.0), y=1.0, timeIndex=Some(1), z=Some(1.0)),
  Obs(i="3", x=Array(1.0, 2.0), y=0.0, timeIndex=Some(1), z=Some(0.0)),
  Obs(i="4", x=Array(2.0, 3.0), y=0.0, timeIndex=Some(1), z=Some(0.0)),
)

// repeat 10 times
// val mediumData = smallData.flatMap(obs => Seq.fill(10)(obs))

val df = smallData.toDS()

smallData = List(Obs(1,[D@2140a028,1.0,Some(1),Some(1.0)), Obs(2,[D@3eada99,1.0,Some(1),Some(1.0)), Obs(3,[D@10fdd2e3,0.0,Some(1),Some(0.0)), Obs(4,[D@ebae6bd,0.0,Some(1),Some(0.0)))
df = [i: string, x: array<double> ... 3 more fields]


[i: string, x: array<double> ... 3 more fields]

In [5]:
val drgu = new DRGU()
drgu.fit(df, maxIter = 10, tol = 1e-6, lambda = 0.1, dampingOnly = false, verbose = true)

Iteration 0: diff = 0.844447643301517
Iteration 1: diff = 0.260565451986891
Iteration 2: diff = 0.08360083381668763
Iteration 3: diff = 0.03981630566128354
Iteration 4: diff = 0.017483679231299197
Iteration 5: diff = 0.008004914089777037
Iteration 6: diff = 0.0036010759921627027
Iteration 7: diff = 0.001633315589146206
Iteration 8: diff = 7.38105188025919E-4
Iteration 9: diff = 3.3411040815950016E-4
DRGU did not converge after 10 iterations
Final step norm: 1.5112464249458654E-4
Final parameter estimates:
Map(delta -> DenseVector(0.5811975004227125), beta -> DenseVector(1.2959982095515519E-18, 2.002439981033718E-18, 3.3179541463590074E-18), gamma -> DenseVector(0.06568275629986627, 0.0945547752347141, 0.1602375315345804, 0.0945547752347141, 0.1602375315345804))
Final variance estimate:
0.20974479159395934     1.7131591180048592E-13  ... (9 total)
1.7131591180048587E-13  2.370370370370521       ...
1.8947238264129917E-13  -1.1851851851851376     ...
-1.8235779575066E-13    1.18518518518

drgu = robustinfer.DRGU@4fee8ec3


robustinfer.DRGU@4fee8ec3

In [6]:
drgu.summary().show()

+---------+--------------------+-------------------+--------------------+--------------------+
|parameter|            estimate|          std_error|             z_score|             p_value|
+---------+--------------------+-------------------+--------------------+--------------------+
|    delta|  1.0811975004227126|0.22898951482216348|  2.5380965625176275|0.011145722675828695|
|   beta_0|1.295998209551551...| 0.7698003589195255|1.683551059096134...|                 1.0|
|   beta_1|2.002439981033718...| 0.3849001794597322|5.202491679386735...|                 1.0|
|   beta_2|3.317954146359007...| 0.3849001794597492|8.620297738016466...|                 1.0|
|  gamma_0| 0.06568275629986627| 23.508931260089764|0.002793949055922...|  0.9977707540849154|
|  gamma_1|  0.0945547752347141| 17.573165658745822|0.005380634148159642|  0.9957068958009683|
|  gamma_2|  0.1602375315345804|  6.550435737530165| 0.02446211793461511|  0.9804840001749886|
|  gamma_3|  0.0945547752347141| 17.57316565874582

## simulate data

In [7]:
def generateData(
    numClusters: Int,
    numObsPerCluster: Int,
    p: Int,
    etaTrue: Array[Double],
    betaTrue: Array[Double],
): Dataset[Obs] = {
  val random = new Random()
  val data = (1 to numClusters).flatMap { clusterId =>
    (1 to numObsPerCluster).map { obsId =>
      val w = Array.fill(p)(random.nextDouble()) // p covariates
      val Wt = 1.0 +: w                          // Add intercept
      val piTrue = 1.0 / (1.0 + math.exp(-(Wt zip etaTrue).map { case (w, e) => w * e }.sum))
      val z = if (random.nextDouble() < piTrue) 1.0 else 0.0
      val X = 1.0 +: z +: w
      val error = random.nextGaussian()
      val y = (X zip betaTrue).map { case (x, b) => x * b }.sum + error

      Obs(
        i = s"c$clusterId",
        x = w,
        y = y,
        timeIndex = Some(obsId),
        z = Some(z)
      )
    }
  }

  data.toDS()
}

generateData: (numClusters: Int, numObsPerCluster: Int, p: Int, etaTrue: Array[Double], betaTrue: Array[Double])org.apache.spark.sql.Dataset[robustinfer.Obs]


In [8]:
val p = 1
val etaTrue = Array(0.0, 0.0)
val betaTrue = Array(0.0, 0.0, 0.0)

val numClusters = 500
val numObsPerCluster = 1

require(etaTrue.length == p + 1, "Length of eta must be p + 1")
require(betaTrue.length == p + 2, "Length of beta must be p + 2")

p = 1
etaTrue = Array(0.0, 0.0)
betaTrue = Array(0.0, 0.0, 0.0)
numClusters = 500
numObsPerCluster = 1


1

In [9]:
val df: Dataset[Obs] = generateData(numClusters, numObsPerCluster, p, etaTrue, betaTrue)

df = [i: string, x: array<double> ... 3 more fields]


[i: string, x: array<double> ... 3 more fields]

In [10]:
val drgu = new DRGU()
drgu.fit(df, maxIter = 10, tol = 1e-8, verbose = true)
// drgu.fit(df, maxIter = 10, tol = 1e-8, lambda = 0.000, dampingOnly = false, verbose = true)

Iteration 0: diff = 0.6247105454385786
Iteration 1: diff = 0.06643541772656655
Iteration 2: diff = 6.997774828657227E-4
Iteration 3: diff = 2.364717829525928E-5
Iteration 4: diff = 5.117269198838825E-7
Iteration 5: diff = 1.3520677843538785E-8
Iteration 6: diff = 3.315233526189399E-10
Final step norm: 8.413682907271312E-12
Final parameter estimates:
Map(delta -> DenseVector(-0.04762084173609247), beta -> DenseVector(-0.18380993226349596, 0.4621081873812349), gamma -> DenseVector(0.07343607999534078, -0.219265241113062, -0.2811774431225924))
Final variance estimate:
1.9970591533283666   -1.7451750490849476  4.053136100700786    ... (6 total)
-1.7451750490849478  63.81375676996433    -108.03438971623967  ...
4.053136100700786    -108.03438971623977  209.4285065751463    ...
234.9148501815796    -202.7751470942184   471.93394992200274   ...
-2568.6931696702914  2287.7541493385725   -5289.614528625848   ...
2568.55575362039     -2291.5120779023287  5296.3297907376655   ...


drgu = robustinfer.DRGU@6d02849d


robustinfer.DRGU@6d02849d

In [11]:
drgu.summary().show()

+---------+--------------------+-------------------+--------------------+------------------+
|parameter|            estimate|          std_error|             z_score|           p_value|
+---------+--------------------+-------------------+--------------------+------------------+
|    delta|  0.4523791582639075|0.06319903722887504| -0.7535058099640631|0.4511460213118599|
|   beta_0|-0.18380993226349596| 0.3572499314764506| -0.5145135549889179|0.6068929858761503|
|   beta_1|  0.4621081873812349| 0.6471916355688573|   0.714020642394519|0.4752144151799633|
|  gamma_0| 0.07343607999534078|  7.485554376153135|0.009810372926992212|0.9921725804623029|
|  gamma_1|  -0.219265241113062|  81.59025412368406|-0.00268739500162...|0.9978557716003964|
|  gamma_2| -0.2811774431225924|  81.57908661525863|-0.00344668535514...| 0.997249948414126|
+---------+--------------------+-------------------+--------------------+------------------+



## Simulation

In [13]:
// simulation 
// loop 100 times of generating data
val numSim = 100
val deltaPValues = scala.collection.mutable.ArrayBuffer[Double]()
val deltaVValues = scala.collection.mutable.ArrayBuffer[Double]()

for (sim <- 1 to numSim) {
  val df: Dataset[Obs] = generateData(numClusters, numObsPerCluster, p, etaTrue, betaTrue)

  // Initialize theta
  val drgu = new DRGU()
  drgu.fit(df, maxIter = 10, tol = 1e-5, lambda = 0.0001, dampingOnly = false, verbose = false)
  val summaryDF = drgu.summary()
  // extract p-value and delta
  val delta = summaryDF.filter($"parameter" === "delta").select("estimate").as[Double].head()
  val deltaPValue = summaryDF.filter($"parameter" === "delta").select("p_value").as[Double].head()

  // store to vector
  deltaPValues += deltaPValue
  deltaVValues += delta

  if (sim % 10 == 0) {
    println(s"Simulation $sim: delta = $delta, delta p-value = $deltaPValue")
  }
}

Simulation 10: delta = 0.468510276729177, delta p-value = 0.8784921611674212
Simulation 20: delta = 0.3335896235040394, delta p-value = 0.38764957017714696
Simulation 30: delta = 0.5019129321523381, delta p-value = 0.992455270969961
Simulation 40: delta = 0.5134019025453815, delta p-value = 0.9475576778193036
Simulation 50: delta = 0.34724991867686983, delta p-value = 0.46435188922362336
Simulation 60: delta = 0.6982807553977572, delta p-value = 0.319013961802191
Simulation 70: delta = 0.48340617719946744, delta p-value = 0.9369791222960315
Simulation 80: delta = 0.4157976859317144, delta p-value = 0.6888666046168157
Simulation 90: delta = 0.6790060134305127, delta p-value = 0.3419562748059737
Simulation 100: delta = 0.6334185292325989, delta p-value = 0.504816287922192


numSim = 100
deltaPValues = ArrayBuffer(0.6048816590533084, 0.45813220934599874, 0.7941263250449664, 0.9247320879865284, 0.8608597097609665, 0.7129577012024932, 0.8828752434187526, 0.46384654940257297, 0.8217136773246534, 0.8784921611674212, 0.9746206006967386, 0.6610698210705261, 0.8253553313457851, 0.6456104199891679, 0.6949599383698684, 0.7805270220178946, 0.7677659629961777, 0.7931387680732023, 0.9512066694377217, 0.38764957017714696, 0.802381694085307, 0.8903654636135596, 0.7706222481324925, 0.5531931841775035, 0.8069519624879402, 0.8473026685539675, 0.6806558500032891, 0.750980083283828, 0.6268917810632908, 0.992455270969961, 0.5723276147575533, 0.8014505169466504, 0.6965392990480146, 0.4458833828487885, 0.7634837987059153, 0.7763...


ArrayBuffer(0.6048816590533084, 0.45813220934599874, 0.7941263250449664, 0.9247320879865284, 0.8608597097609665, 0.7129577012024932, 0.8828752434187526, 0.46384654940257297, 0.8217136773246534, 0.8784921611674212, 0.9746206006967386, 0.6610698210705261, 0.8253553313457851, 0.6456104199891679, 0.6949599383698684, 0.7805270220178946, 0.7677659629961777, 0.7931387680732023, 0.9512066694377217, 0.38764957017714696, 0.802381694085307, 0.8903654636135596, 0.7706222481324925, 0.5531931841775035, 0.8069519624879402, 0.8473026685539675, 0.6806558500032891, 0.750980083283828, 0.6268917810632908, 0.992455270969961, 0.5723276147575533, 0.8014505169466504, 0.6965392990480146, 0.4458833828487885, 0.7634837987059153, 0.7763...

In [15]:
// quantiles of delta and deltaPValues
val deltaQuantiles = deltaVValues.sorted.grouped(numSim / 10).map(_.head).toArray
val deltaPValuesQuantiles = deltaPValues.sorted.grouped(numSim / 10).map(_.head).toArray

// type i error
val typeIError = deltaPValues.count(_ < 0.05).toDouble / numSim
println(s"Type I error rate: $typeIError")

Type I error rate: 0.0


deltaQuantiles = Array(0.29517009751097983, 0.3715933940572218, 0.4205434984887063, 0.4485535410627366, 0.4731282820240386, 0.5019129321523381, 0.5259584157467033, 0.5456702212611314, 0.5711437007798136, 0.6126893150157692)
deltaPValuesQuantiles = Array(0.21208435992588992, 0.46128603376230926, 0.5531931841775035, 0.6546806400595375, 0.7263733011083793, 0.7721047334894493, 0.8069519624879402, 0.8731221018713722, 0.9016219340400595, 0.9369791222960315)
typeIError = 0.0


0.0